# 05 — Stacking 11-base (ElasticNetCV meta + StandardScaler)

**목적**: 모든 가용 base 모델의 OOF/val/test 예측을 모아 ElasticNet meta-learner로 stacking. 모듈 수정 0, csv input만 사용.

**Base 11종**:
| 그룹 | 모델 |
|---|---|
| ZIT only | `zit_only` (final) |
| BagZIT 변형 6종 | `bag_zit_combined_best` (HP#11+PP#4+pos), `bag_zit_combined_best_xy` (+xy), `bag_zit_pp_hpo`, `bag_zit_hpo`, `bag_zit_fixed_ge` |
| reg_only 4종 | `lgbm`, `enet`, `et`, `catboost` |

**Meta-learner**: `ElasticNetCV(StandardScaler, l1_ratio grid, alpha grid, KFold=5)`
- 음수 weight 허용 (positive=False) → 일부 모델은 corrector 역할
- Lasso성으로 redundant 모델은 0으로 죽임
- 정규화 alpha + l1_ratio 자동 선택

**비교 baseline**: SLSQP blending (Σw=1, w≥0)

**격리**: `4_output/final/stacking_11base/` 신규. 기존 모듈/노트북 무수정.

**선행 조건**: 11종 base 산출물이 모두 존재해야 함 (`oof_unit.csv`, `val_unit.csv`, `test_unit.csv`).

## 1. 환경 + import

In [1]:
import os, sys, json

%run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR,
)
from utils.data import load_all

from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from scipy.optimize import minimize

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. Base pool 정의 + 자동 탐지

각 산출물 디렉토리에 `oof_unit.csv`/`val_unit.csv`/`test_unit.csv` 모두 존재하면 pool 포함. 누락 모델은 자동 제외.

In [2]:
POOL_PATHS = {
    # ZIT only
    'zit_only':                 os.path.join(OUTPUT_DIR, 'final', 'zit_only'),
    # BagZIT 변형 6종
    'bag_zit_combined_best':    os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_combined_best'),
    'bag_zit_combined_best_xy': os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_combined_best_xy'),
    'bag_zit_pp_hpo':           os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_pp_hpo'),
    'bag_zit_hpo':              os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_hpo'),
    'bag_zit_fixed_ge':         os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_fixed_ge'),
    # reg_only 4종
    'lgbm':                     os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'lgbm'),
    'enet':                     os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'enet'),
    'et':                       os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'et'),
    'catboost':                 os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'catboost'),
}

REQUIRED = ['oof_unit.csv', 'val_unit.csv', 'test_unit.csv']
available, missing = {}, {}
for name, base in POOL_PATHS.items():
    have = all(os.path.exists(os.path.join(base, f)) for f in REQUIRED)
    if have:
        available[name] = base
    else:
        miss = [f for f in REQUIRED if not os.path.exists(os.path.join(base, f))]
        missing[name] = miss

print(f'=== Pool 자동 탐지 ===')
print(f'  사용 가능 ({len(available)}/{len(POOL_PATHS)}):')
for n in available:
    print(f'    ✓ {n}')
if missing:
    print(f'\n  누락 ({len(missing)}):')
    for n, miss in missing.items():
        print(f'    ✗ {n}  (missing: {miss})')

if len(available) < 2:
    raise RuntimeError(f'stacking 가능한 base < 2개')

=== Pool 자동 탐지 ===
  사용 가능 (10/10):
    ✓ zit_only
    ✓ bag_zit_combined_best
    ✓ bag_zit_combined_best_xy
    ✓ bag_zit_pp_hpo
    ✓ bag_zit_hpo
    ✓ bag_zit_fixed_ge
    ✓ lgbm
    ✓ enet
    ✓ et
    ✓ catboost


## 3. OOF/val/test 로드 + 정합 검증

각 csv는 `[ufs_serial, pred, health]` 컬럼. 모든 base 모델이 동일 ufs_serial set 가져야 정합.

In [3]:
_, ys = load_all()
y_train_unit = ys['train'][[KEY_COL, TARGET_COL]].copy()
y_val_unit   = ys['validation'][[KEY_COL, TARGET_COL]].copy()
y_test_unit  = ys['test'][[KEY_COL, TARGET_COL]].copy()

# csv 내장 health 사용 (CLIP_Y_EXTREME 적용 상태 — 공정 비교)
first_name = next(iter(available))
first_oof = pd.read_csv(os.path.join(available[first_name], 'oof_unit.csv'))
first_val = pd.read_csv(os.path.join(available[first_name], 'val_unit.csv'))
first_test = pd.read_csv(os.path.join(available[first_name], 'test_unit.csv'))
y_oof  = first_oof.set_index(KEY_COL)['health']
y_val  = first_val.set_index(KEY_COL)['health']
y_test = first_test.set_index(KEY_COL)['health']

print(f'y counts: oof={len(y_oof):,}, val={len(y_val):,}, test={len(y_test):,}')
print(f'y_train (csv health, clipped) max: oof={y_oof.max():.6f}, val={y_val.max():.6f}, test={y_test.max():.6f}')

# 모든 base 모델 pred 로드
oofs, vals, tests = {}, {}, {}
for n, base in available.items():
    oofs[n]  = pd.read_csv(os.path.join(base, 'oof_unit.csv')).set_index(KEY_COL)['pred']
    vals[n]  = pd.read_csv(os.path.join(base, 'val_unit.csv')).set_index(KEY_COL)['pred']
    tests[n] = pd.read_csv(os.path.join(base, 'test_unit.csv')).set_index(KEY_COL)['pred']

# 정합 검증
def _check_index(d, ref, label):
    for n, s in d.items():
        miss = ref.difference(s.index)
        extra = s.index.difference(ref)
        if len(miss) or len(extra):
            raise ValueError(f'{label}/{n}: missing={len(miss)}, extra={len(extra)}')

_check_index(oofs,  y_oof.index,  'oof')
_check_index(vals,  y_val.index,  'val')
_check_index(tests, y_test.index, 'test')

# 정렬된 prediction matrix
P_oof  = pd.DataFrame({n: oofs[n].reindex(y_oof.index)   for n in available})
P_val  = pd.DataFrame({n: vals[n].reindex(y_val.index)   for n in available})
P_test = pd.DataFrame({n: tests[n].reindex(y_test.index) for n in available})

print(f'\n[정합 OK] P_oof={P_oof.shape}, P_val={P_val.shape}, P_test={P_test.shape}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
y counts: oof=26,187, val=8,727, test=8,729
y_train (csv health, clipped) max: oof=1.000000, val=0.172211, test=0.602242

[정합 OK] P_oof=(26187, 10), P_val=(8727, 10), P_test=(8729, 10)


## 4. 단일 base RMSE + residual correlation

In [4]:
def _rmse(p, y):
    return float(np.sqrt(np.mean((np.asarray(p) - np.asarray(y)) ** 2)))

rows = []
for n in available:
    rows.append({
        'model': n,
        'oof':   _rmse(P_oof[n].values,  y_oof.values),
        'val':   _rmse(P_val[n].values,  y_val.values),
        'test':  _rmse(P_test[n].values, y_test.values),
    })
single_df = pd.DataFrame(rows).sort_values('val')
print('=== 단일 base RMSE (val 오름차순) ===')
print(single_df.to_string(index=False, float_format='%.6f'))

# residual corr
R_oof  = P_oof.subtract(y_oof,  axis=0)
R_test = P_test.subtract(y_test, axis=0)
print('\n=== Residual correlation (OOF) ===')
print(R_oof.corr().round(4).to_string())
print('\n=== Residual correlation (test) ===')
print(R_test.corr().round(4).to_string())

# 가장 보완적인 페어
names_list = list(available)
min_pair = (None, None, 1.0)
corr_oof = R_oof.corr()
for i, a in enumerate(names_list):
    for b in names_list[i+1:]:
        c = corr_oof.loc[a, b]
        if c < min_pair[2]:
            min_pair = (a, b, c)
print(f'\n가장 낮은 OOF residual corr: {min_pair[0]} ↔ {min_pair[1]} = {min_pair[2]:.4f}')

=== 단일 base RMSE (val 오름차순) ===
                   model      oof      val     test
                zit_only 0.008246 0.005709 0.008414
   bag_zit_combined_best 0.008251 0.005710 0.008412
             bag_zit_hpo 0.008244 0.005711 0.008417
bag_zit_combined_best_xy 0.008251 0.005712 0.008411
          bag_zit_pp_hpo 0.008245 0.005712 0.008414
        bag_zit_fixed_ge 0.008253 0.005717 0.008412
                catboost 0.008259 0.005731 0.008430
                    lgbm 0.008257 0.005731 0.008429
                      et 0.008269 0.005758 0.008452
                    enet 0.008286 0.005781 0.008459

=== Residual correlation (OOF) ===
                          zit_only  bag_zit_combined_best  bag_zit_combined_best_xy  bag_zit_pp_hpo  bag_zit_hpo  bag_zit_fixed_ge    lgbm    enet      et  catboost
zit_only                    1.0000                 0.9990                    0.9989          0.9994       0.9991            0.9971  0.9983  0.9945  0.9959    0.9982
bag_zit_combined_best       0.

## 5. Blending baseline (SLSQP — Σw=1, w≥0)

Stacking 효과 비교용. 제약 있는 가중평균.

In [5]:
K = len(available)
P_oof_arr  = P_oof.values
P_val_arr  = P_val.values
P_test_arr = P_test.values
y_oof_arr  = y_oof.values
y_val_arr  = y_val.values
y_test_arr = y_test.values

res = minimize(
    lambda w: _rmse(P_oof_arr @ w, y_oof_arr),
    np.full(K, 1.0/K),
    method='SLSQP',
    bounds=[(0.0, 1.0)] * K,
    constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}],
    options={'ftol': 1e-9, 'maxiter': 500},
)
w_blend = res.x

blend_oof  = P_oof_arr  @ w_blend
blend_val  = P_val_arr  @ w_blend
blend_test = P_test_arr @ w_blend

rmse_blend_oof  = _rmse(blend_oof,  y_oof_arr)
rmse_blend_val  = _rmse(blend_val,  y_val_arr)
rmse_blend_test = _rmse(blend_test, y_test_arr)

print(f'=== Blending SLSQP (converged={res.success}) ===')
for n, w in sorted(zip(available, w_blend), key=lambda x: -x[1]):
    bar = '█' * int(w * 40)
    print(f'  {n:28s}: {w:.4f}  {bar}')
print(f'\n  OOF RMSE:  {rmse_blend_oof:.6f}')
print(f'  val RMSE:  {rmse_blend_val:.6f}')
print(f'  test RMSE: {rmse_blend_test:.6f}')

=== Blending SLSQP (converged=True) ===
  zit_only                    : 0.1000  ████
  bag_zit_combined_best       : 0.1000  ████
  bag_zit_combined_best_xy    : 0.1000  ████
  bag_zit_pp_hpo              : 0.1000  ████
  bag_zit_hpo                 : 0.1000  ████
  bag_zit_fixed_ge            : 0.1000  ████
  lgbm                        : 0.1000  ████
  enet                        : 0.1000  ████
  et                          : 0.1000  ████
  catboost                    : 0.1000  ████

  OOF RMSE:  0.008247
  val RMSE:  0.005716
  test RMSE: 0.008417


## 6. Stacking — ElasticNetCV meta + StandardScaler

**ElasticNetCV**가 자동으로:
- `alpha` (정규화 강도) — `np.logspace(-6, 0, 30)` 그리드
- `l1_ratio` (Lasso성) — `[0.1, 0.3, 0.5, 0.7, 0.9, 1.0]` 그리드

5-fold CV로 best 자동 선택. `positive=False` 라 음수 weight 허용. 음수 예측은 0으로 clip (health ≥ 0).

In [6]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('enet',   ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
        alphas=np.logspace(-6, 0, 30),
        cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
        random_state=SEED,
        n_jobs=-1,
        max_iter=20000,
        positive=False,
    )),
])
pipe.fit(P_oof_arr, y_oof_arr)

en = pipe.named_steps['enet']
best_alpha    = float(en.alpha_)
best_l1_ratio = float(en.l1_ratio_)
stack_coef    = en.coef_
stack_intercept = float(en.intercept_)

print(f'=== ElasticNetCV best ===')
print(f'  alpha    : {best_alpha:.6e}')
print(f'  l1_ratio : {best_l1_ratio:.4f}')
print(f'  intercept: {stack_intercept:+.6f}')
print(f'  coef (scaled space, abs 정렬):')
for n, c in sorted(zip(available, stack_coef), key=lambda x: -abs(x[1])):
    bar = '█' * int(min(abs(c) * 1500, 40))
    sign = '+' if c >= 0 else '-'
    print(f'    {n:28s}: {sign}{abs(c):.5f}  {bar}')

stack_oof  = np.clip(pipe.predict(P_oof_arr),  0, None)
stack_val  = np.clip(pipe.predict(P_val_arr),  0, None)
stack_test = np.clip(pipe.predict(P_test_arr), 0, None)

rmse_stack_oof  = _rmse(stack_oof,  y_oof_arr)
rmse_stack_val  = _rmse(stack_val,  y_val_arr)
rmse_stack_test = _rmse(stack_test, y_test_arr)

neg_pre_clip_oof  = (pipe.predict(P_oof_arr)  < 0).mean()
neg_pre_clip_val  = (pipe.predict(P_val_arr)  < 0).mean()
neg_pre_clip_test = (pipe.predict(P_test_arr) < 0).mean()

print(f'\n=== Stacking RMSE ===')
print(f'  OOF RMSE:  {rmse_stack_oof:.6f}')
print(f'  val RMSE:  {rmse_stack_val:.6f}')
print(f'  test RMSE: {rmse_stack_test:.6f}')
print(f'  음수 clip 비율: oof={neg_pre_clip_oof:.1%}, val={neg_pre_clip_val:.1%}, test={neg_pre_clip_test:.1%}')

=== ElasticNetCV best ===
  alpha    : 4.175319e-06
  l1_ratio : 0.9000
  intercept: +0.002515
  coef (scaled space, abs 정렬):
    bag_zit_hpo                 : +0.00077  █
    bag_zit_fixed_ge            : +0.00046  
    zit_only                    : +0.00040  
    lgbm                        : -0.00026  
    bag_zit_combined_best_xy    : -0.00025  
    et                          : +0.00014  
    catboost                    : -0.00013  
    enet                        : -0.00008  
    bag_zit_combined_best       : +0.00000  
    bag_zit_pp_hpo              : +0.00000  

=== Stacking RMSE ===
  OOF RMSE:  0.008237
  val RMSE:  0.005701
  test RMSE: 0.008408
  음수 clip 비율: oof=0.2%, val=0.1%, test=0.1%


## 7. 종합 비교 표

In [7]:
best_oof_row  = single_df.sort_values('oof').iloc[0]
best_val_row  = single_df.sort_values('val').iloc[0]
best_test_row = single_df.sort_values('test').iloc[0]

comparison = pd.DataFrame([
    {'method': f'best single (OOF) [{best_oof_row["model"]}]',
     'oof': best_oof_row['oof'], 'val': best_oof_row['val'], 'test': best_oof_row['test']},
    {'method': f'best single (val) [{best_val_row["model"]}]',
     'oof': best_val_row['oof'], 'val': best_val_row['val'], 'test': best_val_row['test']},
    {'method': f'best single (test)[{best_test_row["model"]}]',
     'oof': best_test_row['oof'], 'val': best_test_row['val'], 'test': best_test_row['test']},
    {'method': f'Blending SLSQP ({K} base)',
     'oof': rmse_blend_oof, 'val': rmse_blend_val, 'test': rmse_blend_test},
    {'method': f'Stacking ElasticNet ({K} base)',
     'oof': rmse_stack_oof, 'val': rmse_stack_val, 'test': rmse_stack_test},
])
print('=' * 95)
print(f'  Stacking 11-base — 종합 비교 (사용 가능 base = {K}종)')
print('=' * 95)
print(comparison.to_string(index=False, float_format='%.6f'))
print('=' * 95)

# Δ
for label, oof, val, test in [
    ('Blending  vs single best ', rmse_blend_oof, rmse_blend_val, rmse_blend_test),
    ('Stacking  vs single best ', rmse_stack_oof, rmse_stack_val, rmse_stack_test),
]:
    d_oof  = oof  - best_oof_row['oof']
    d_val  = val  - best_val_row['val']
    d_test = test - best_test_row['test']
    print(f'  {label}: Δoof={d_oof:+.6f}  Δval={d_val:+.6f}  Δtest={d_test:+.6f}')

  Stacking 11-base — 종합 비교 (사용 가능 base = 10종)
                                      method      oof      val     test
             best single (OOF) [bag_zit_hpo] 0.008244 0.005711 0.008417
                best single (val) [zit_only] 0.008246 0.005709 0.008414
best single (test)[bag_zit_combined_best_xy] 0.008251 0.005712 0.008411
                    Blending SLSQP (10 base) 0.008247 0.005716 0.008417
               Stacking ElasticNet (10 base) 0.008237 0.005701 0.008408
  Blending  vs single best : Δoof=+0.000002  Δval=+0.000007  Δtest=+0.000006
  Stacking  vs single best : Δoof=-0.000007  Δval=-0.000008  Δtest=-0.000004


## 8. 산출물 저장 (`4_output/final/stacking_11base/`)

In [8]:
OUT_DIR = os.path.join(OUTPUT_DIR, 'final', 'stacking_11base')
os.makedirs(OUT_DIR, exist_ok=True)

def _save_unit(pred_arr, ids, y_arr, fname):
    out = pd.DataFrame({
        KEY_COL: ids,
        'pred':  pred_arr,
        'health': y_arr.reindex(ids).values,
    })
    out.to_csv(os.path.join(OUT_DIR, fname), index=False)

# stacking 산출물
_save_unit(stack_oof,  y_oof.index,  y_oof,  'oof_unit_stack.csv')
_save_unit(stack_val,  y_val.index,  y_val,  'val_unit_stack.csv')
_save_unit(stack_test, y_test.index, y_test, 'test_unit_stack.csv')

# blending baseline 산출물
_save_unit(blend_oof,  y_oof.index,  y_oof,  'oof_unit_blend.csv')
_save_unit(blend_val,  y_val.index,  y_val,  'val_unit_blend.csv')
_save_unit(blend_test, y_test.index, y_test, 'test_unit_blend.csv')

single_df.to_csv(os.path.join(OUT_DIR, 'single_base_rmse.csv'), index=False)
R_oof.corr().to_csv(os.path.join(OUT_DIR, 'residual_corr_oof.csv'))
R_test.corr().to_csv(os.path.join(OUT_DIR, 'residual_corr_test.csv'))
comparison.to_csv(os.path.join(OUT_DIR, 'comparison.csv'), index=False)

meta = {
    'pool_models':     list(available.keys()),
    'pool_paths':      {k: available[k] for k in available},
    'missing_models':  list(missing.keys()),
    'meta_learner': {
        'type':       'ElasticNetCV(StandardScaler)',
        'alpha':      best_alpha,
        'l1_ratio':   best_l1_ratio,
        'coef':       {n: float(c) for n, c in zip(available, stack_coef)},
        'intercept':  stack_intercept,
        'positive':   False,
        'max_iter':   20000,
        'cv':         '5-fold KFold(shuffle=True, random_state=SEED)',
    },
    'blending_slsqp': {
        'weights':    {n: float(w) for n, w in zip(available, w_blend)},
        'converged':  bool(res.success),
        'rmse_oof':   rmse_blend_oof,
        'rmse_val':   rmse_blend_val,
        'rmse_test':  rmse_blend_test,
    },
    'stacking': {
        'rmse_oof':  rmse_stack_oof,
        'rmse_val':  rmse_stack_val,
        'rmse_test': rmse_stack_test,
    },
    'single_base': single_df.to_dict(orient='records'),
    'min_residual_corr_pair': {
        'a': min_pair[0], 'b': min_pair[1], 'corr_oof': float(min_pair[2]),
    },
    'SEED': int(SEED),
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\stacking_11base
  comparison.csv                         0.5 KB
  meta.json                              4.3 KB
  oof_unit_blend.csv                   952.1 KB
  oof_unit_stack.csv                   970.3 KB
  residual_corr_oof.csv                  1.9 KB
  residual_corr_test.csv                 1.9 KB
  single_base_rmse.csv                   0.8 KB
  test_unit_blend.csv                  317.2 KB
  test_unit_stack.csv                  323.6 KB
  val_unit_blend.csv                   317.1 KB
  val_unit_stack.csv                   323.4 KB


## 9. 요약

In [9]:
print('=' * 95)
print(f' Stacking {K}-base — 결과 요약')
print('=' * 95)
print(comparison.to_string(index=False, float_format='%.6f'))
print('-' * 95)
print(f'  Meta best alpha    : {best_alpha:.4e}')
print(f'  Meta best l1_ratio : {best_l1_ratio:.4f}')
print(f'  Meta intercept     : {stack_intercept:+.6f}')
nonzero = sum(1 for c in stack_coef if abs(c) > 1e-8)
neg     = sum(1 for c in stack_coef if c < -1e-8)
print(f'  활성 모델 수        : {nonzero}/{K} (음수 weight {neg}개 = corrector 역할)')
print(f'  최저 residual corr : {min_pair[0]} ↔ {min_pair[1]} = {min_pair[2]:.4f}')
print('=' * 95)
print(f'  → val/test RMSE 가 단일 best보다 낮으면 stacking 효과 입증.')
print(f'  → meta가 일부 모델을 0 또는 음수 weight 로 처리한 건 redundant/corrector 판단.')

 Stacking 10-base — 결과 요약
                                      method      oof      val     test
             best single (OOF) [bag_zit_hpo] 0.008244 0.005711 0.008417
                best single (val) [zit_only] 0.008246 0.005709 0.008414
best single (test)[bag_zit_combined_best_xy] 0.008251 0.005712 0.008411
                    Blending SLSQP (10 base) 0.008247 0.005716 0.008417
               Stacking ElasticNet (10 base) 0.008237 0.005701 0.008408
-----------------------------------------------------------------------------------------------
  Meta best alpha    : 4.1753e-06
  Meta best l1_ratio : 0.9000
  Meta intercept     : +0.002515
  활성 모델 수        : 8/10 (음수 weight 4개 = corrector 역할)
  최저 residual corr : bag_zit_fixed_ge ↔ enet = 0.9914
  → val/test RMSE 가 단일 best보다 낮으면 stacking 효과 입증.
  → meta가 일부 모델을 0 또는 음수 weight 로 처리한 건 redundant/corrector 판단.
